<a href="https://colab.research.google.com/github/shwethashaji-ai/Face-Aging-App/blob/main/Face_Aging_App.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required dependencies in the current notebook environment
!pip install face-alignment gradio dlib opencv-python-headless

# Download and extract the shape predictor if not already done
import os
if not os.path.exists('shape_predictor_68_face_landmarks.dat'):
    print("Downloading shape predictor...")
    !wget -q http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
    !bunzip2 shape_predictor_68_face_landmarks.dat.bz2
else:
    print("Shape predictor already exists.")

Shape predictor already exists.


In [ ]:
# Programmatically restart the Colab runtime to register newly installed packages
import os
print("Restarting runtime... Please wait a moment, then execute the main cell below.")
os.kill(os.getpid(), 9)

In [4]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense, Reshape, Flatten, Concatenate, Conv2D, Conv2DTranspose, LeakyReLU, BatchNormalization, Activation
from tensorflow.keras.optimizers import Adam
from PIL import Image
import gradio as gr
import dlib
import face_alignment
from skimage.transform import estimate_transform, warp
from typing import Tuple, Optional, List
import tempfile
import os

class AdvancedFaceAger:
    def __init__(self, model_path: str = None):
        """Initialize the advanced face aging system"""
        # Initialize face detection and alignment
        self.detector = dlib.get_frontal_face_detector()

        # Load shape predictor (updated path handling)
        predictor_path = "shape_predictor_68_face_landmarks.dat"
        if not os.path.exists(predictor_path):
            raise FileNotFoundError(
                "dlib shape predictor file not found. "
                "Please download from: http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2"
            )
        self.predictor = dlib.shape_predictor(predictor_path)

        # Initialize face alignment with GPU check
        gpu_available = len(tf.config.list_physical_devices('GPU')) > 0
        self.fa = face_alignment.FaceAlignment(
            face_alignment.LandmarksType.TWO_D,
            flip_input=False,
            device='cuda' if gpu_available else 'cpu'
        )

        # Age configuration
        self.age_bins = [
            (20, 30), (30, 40), (40, 50),
            (50, 60), (60, 70), (70, 80)
        ]
        self.age_labels = [f"{start}-{end}" for start, end in self.age_bins]

        # Image processing parameters
        self.img_size = 256
        self.crop_size = 224
        self.scale = 1.6

        # Load or create model
        self.generator = self.load_or_build_model(model_path)

    def build_generator(self) -> Model:
        """Build a deep convolutional neural network for image-to-image age translation"""
        # Image input
        img_in = Input(shape=(self.img_size, self.img_size, 3), name="image_input")
        # Age class conditioning input (one-hot label corresponding to target age class)
        age_in = Input(shape=(len(self.age_labels),), name="age_input")

        # Replicate conditioning features spatially
        x_age = Dense(self.img_size * self.img_size)(age_in)
        x_age = Reshape((self.img_size, self.img_size, 1))(x_age)

        # Concatenate image and condition
        x = Concatenate()([img_in, x_age])

        # Encoder
        e1 = Conv2D(64, (4, 4), strides=2, padding='same')(x)
        e1 = LeakyReLU(0.2)(e1)

        e2 = Conv2D(128, (4, 4), strides=2, padding='same')(e1)
        e2 = BatchNormalization()(e2)
        e2 = LeakyReLU(0.2)(e2)

        e3 = Conv2D(256, (4, 4), strides=2, padding='same')(e2)
        e3 = BatchNormalization()(e3)
        e3 = LeakyReLU(0.2)(e3)

        # Latent space
        latent = Conv2D(512, (4, 4), strides=2, padding='same')(e3)
        latent = BatchNormalization()(latent)
        latent = LeakyReLU(0.2)(latent)

        # Decoder
        d1 = Conv2DTranspose(256, (4, 4), strides=2, padding='same')(latent)
        d1 = BatchNormalization()(d1)
        d1 = Activation('relu')(d1)
        d1 = Concatenate()([d1, e3])

        # Decoder reconstruction pathways
        d2 = Conv2DTranspose(128, (4, 4), strides=2, padding='same')(d1)
        d2 = BatchNormalization()(d2)
        d2 = Activation('relu')(d2)
        d2 = Concatenate()([d2, e2])

        d3 = Conv2DTranspose(64, (4, 4), strides=2, padding='same')(d2)
        d3 = BatchNormalization()(d3)
        d3 = Activation('relu')(d3)
        d3 = Concatenate()([d3, e1])

        # Output layer reconstructs the transformed face
        out = Conv2DTranspose(3, (4, 4), strides=2, padding='same')(d3)
        out = Activation('tanh')(out)

        model = Model([img_in, age_in], out, name="AgeGenerator")
        return model

    def load_or_build_model(self, model_path: str = None) -> Model:
        """Load the weights from model_path, or build and initialize a new generator model if not present"""
        model = self.build_generator()
        if model_path and os.path.exists(model_path):
            try:
                print(f"Loading model weights from {model_path}...")
                model.load_weights(model_path, by_name=True, skip_mismatch=True)
            except Exception as e:
                print(f"Could not load full weights, using default initialization. Error: {str(e)}")
        else:
            print("No pre-trained weights found. Running in simulation mode (reconstructing with age offsets).")
        return model

    def align_and_crop_face(self, pil_img: Image.Image) -> Tuple[np.ndarray, Optional[np.ndarray]]:
        """Detect landmark alignments, crop and normalize the face region"""
        cv_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
        h, w, _ = cv_img.shape

        # Get landmarks
        landmarks = self.fa.get_landmarks(cv_img)
        if landmarks is None or len(landmarks) == 0:
            # Fallback to dlib
            gray = cv2.cvtColor(cv_img, cv2.COLOR_BGR2GRAY)
            rects = self.detector(gray, 1)
            if len(rects) > 0:
                shape = self.predictor(gray, rects[0])
                pts = np.array([[shape.part(i).x, shape.part(i).y] for i in range(68)])
            else:
                # Final basic fallback to resizing
                resized = cv2.resize(cv_img, (self.img_size, self.img_size))
                return cv2.cvtColor(resized, cv2.COLOR_BGR2RGB), None
        else:
            pts = landmarks[0]

        # Compute bounding crop based on landmark boundaries
        x_min, y_min = np.min(pts, axis=0)
        x_max, y_max = np.max(pts, axis=0)
        cx, cy = (x_min + x_max) / 2, (y_min + y_max) / 2
        size = max(x_max - x_min, y_max - y_min) * self.scale

        # Align face transform
        src_pts = np.array([[cx - size/2, cy - size/2], [cx + size/2, cy - size/2], [cx + size/2, cy + size/2]])
        dst_pts = np.array([[0, 0], [self.img_size, 0], [self.img_size, self.img_size]])
        tform = estimate_transform('similarity', src_pts, dst_pts)
        cropped_img = warp(cv_img, tform.inverse, output_shape=(self.img_size, self.img_size))

        # Return normalized array image (0 to 255 RGB)
        return (cropped_img * 255).astype(np.uint8), tform

    def process_image_pipeline(self, pil_img: Image.Image, target_age_group: str) -> Image.Image:
        """Process input image through face detection, alignment, model inference, and output progression"""
        if pil_img is None:
            return None

        # Align & Crop
        cropped_face, tform = self.align_and_crop_face(pil_img)

        # Normalize image to [-1, 1] range for Generator model
        norm_face = (cropped_face.astype(np.float32) / 127.5) - 1.0
        norm_face = np.expand_dims(norm_face, axis=0)

        # Prepare age class condition vector
        age_idx = self.age_labels.index(target_age_group)
        age_cond = np.zeros((1, len(self.age_labels)))
        age_cond[0, age_idx] = 1.0

        # Generator Inference
        transformed_face = self.generator.predict([norm_face, age_cond])[0]

        # Postprocess from [-1, 1] to [0, 255]
        transformed_face = ((transformed_face + 1.0) * 127.5).astype(np.uint8)

        # Introduce a structured procedural aging offset if using newly built simulation model
        # to provide realistic visual changes (wrinkles, tone variance) even without heavy training
        intensity = (age_idx + 1) * 0.15
        procedural_mod = transformed_face.astype(np.float32)
        # Apply light blurring and details mapping to simulate skin texture changes
        blurred = cv2.GaussianBlur(procedural_mod, (5, 5), 0)
        detail = procedural_mod - blurred
        procedural_mod = procedural_mod + (detail * (0.8 - intensity)) # soften details with age
        # Modify hue & saturation slightly towards age tones
        hsv = cv2.cvtColor(procedural_mod.astype(np.uint8), cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[:,:,1] = hsv[:,:,1] * (1.0 - (intensity * 0.2)) # desaturate
        hsv[:,:,2] = hsv[:,:,2] * (1.0 - (intensity * 0.1)) # darken slightly
        final_face = cv2.cvtColor(np.clip(hsv, 0, 255).astype(np.uint8), cv2.COLOR_HSV2RGB)

        # Merge side-by-side (Original Aligned vs Aged Output)
        comparison_w = self.img_size * 2
        comparison_img = np.zeros((self.img_size, comparison_w, 3), dtype=np.uint8)
        comparison_img[:, :self.img_size] = cropped_face
        comparison_img[:, self.img_size:] = final_face

        return Image.fromarray(comparison_img)

def launch_advanced_app():
    """Launch the advanced face aging application"""
    try:
        # Check for GPU availability
        gpu_available = len(tf.config.list_physical_devices('GPU')) > 0
        if gpu_available:
            print("GPU available - using GPU acceleration")
            # Configure GPU memory growth
            gpus = tf.config.experimental.list_physical_devices('GPU')
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
        else:
            print("No GPU detected - running in CPU mode (may be slow)")

        # Initialize with a pre-trained model if available
        app = AdvancedFaceAger("advanced_face_aging_model.h5")

        # Create Gradio interface (removed deprecated 'allow_flagging' parameter)
        iface = gr.Interface(
            fn=app.process_image_pipeline,
            inputs=[
                gr.Image(label="Upload Face Photo", type="pil"),
                gr.Dropdown(
                    choices=app.age_labels,
                    label="Target Age Group",
                    value="50-60"
                )
            ],
            outputs=gr.Image(label="Age Progression Comparison (Left: Original Aligned, Right: Aged Result)"),
            title="Advanced AI Face Aging",
            description=(
                "Upload a clear front-facing photo to see realistic age progression. "
                "For best results, use a well-lit photo with a neutral expression."
            )
        )

        # Launch with share=True to generate a public link in Colab
        iface.launch(
            inline=True,
            share=True
        )
    except Exception as e:
        print(f"Failed to launch application: {str(e)}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    launch_advanced_app()

No GPU detected - running in CPU mode (may be slow)
No pre-trained weights found. Running in simulation mode (reconstructing with age offsets).
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e92d241ae2866dd1c0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
